In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from torch_geometric.utils import degree
from pathlib import Path
import pickle
import json
import torch
import itertools
import scipy
from astropy.table import Table
from scipy.stats import binned_statistic
import random

import matplotlib.patheffects as pe
from easyquery import Query, QueryMaker
from astrocut import FITSCutout
from astropy.coordinates import SkyCoord
import astropy.units as u
import tqdm
base_dir = Path("..").resolve()
results_dir = base_dir / "results"

import cmasher as cmr
from matplotlib.colors import LinearSegmentedColormap

c0, c1, c2, c3, c4 = '#003f5c', '#58508d', '#bc5090', '#ff6361', '#ffa600'
cmap = LinearSegmentedColormap.from_list(
    'jw_cmap', 
    [c0, c1, c2, c3, c4], 
    N=9
)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, median_absolute_error

metrics_mapping = {
    r"$R^2$": lambda p, y: r2_score(y[np.isfinite(y)], p[np.isfinite(y)]),
    r"RMSE": lambda p, y: root_mean_squared_error(p[np.isfinite(y)], y[np.isfinite(y)]),
    r"MAE":lambda p, y: mean_absolute_error(p[np.isfinite(y)], y[np.isfinite(y)]),
    r"NMAD": lambda p, y: 1.4826 * median_absolute_error(p[np.isfinite(y)], y[np.isfinite(y)]),
    r"Bias": lambda p, y: np.mean(p[np.isfinite(y)] - y[np.isfinite(y)]),
    r"Outlier Frac.": lambda p, y: np.mean(np.absolute(p[np.isfinite(y)] - y[np.isfinite(y)]) > 3 * (1.4826 * median_absolute_error(p[np.isfinite(y)], y[np.isfinite(y)]))),
}

In [ ]:
pyg_data_fname = f"{base_dir}/data/processed/galaxy_graphs.pkl"

with open(pyg_data_fname, "rb") as f:
    data_dict = pickle.load(f)

# Show a graph

In [ ]:
galname = "NGC_1566"
datum = data_dict[galname]

node_pos = datum.pos.numpy()
edge_list = datum.edge_index.numpy()
node_colors = datum.y.numpy()

cmap = cmr.get_sub_cmap(cmr.torch, 0.2, 0.9, N=8)


plt.figure(figsize=(7, 6), dpi=300)
scatter = plt.scatter(
    node_pos[:, 0],
    node_pos[:, 1],
    c=node_colors,
    s=50,
    cmap=cmap,
    vmin=6,
    vmax=10,
    edgecolors='none',
    zorder=2
)

plt.gca().set_aspect("equal")

cb = plt.colorbar(scatter, aspect=30)
cb.set_label('log(age/yr)', fontsize=14)

for spine in plt.gca().spines.values():
    spine.set_visible(False)


plt.xlabel('$\\Delta$RA (arcsec)', fontsize=14)
plt.xlim(plt.xlim()[::-1]) # reverse
plt.ylabel('$\\Delta$Dec (arcsec)', fontsize=14)
plt.grid(True, linestyle='-', alpha=0.1)

plt.title(galname.replace("_", " "), fontsize=18)

plt.savefig(results_dir / "figures/NGC1566-graph-no_edges.pdf", dpi=300)

In [ ]:
galname = "NGC_1566"
datum = data_dict[galname]


# select smaller subset
from matplotlib.collections import LineCollection

selection = datum.edge_attr[:, 0] < 15
edge_index = datum.edge_index[:, selection].T.numpy()

node_pos = datum.pos.numpy()
node_colors = datum.y.numpy()

plt.figure(figsize=(6, 5), dpi=300)
lc = LineCollection(node_pos[edge_index], colors='k', alpha=0.2, linewidths=0.2, zorder=0)
plt.gca().add_collection(lc)

cmap = cmr.get_sub_cmap(cmr.torch, 0.2, 0.9, N=8)

scatter = plt.scatter(
    node_pos[:, 0],
    node_pos[:, 1],
    c=node_colors,
    s=20,
    cmap=cmap,
    vmin=6,
    vmax=10,
    edgecolors='none',
    zorder=2
)

plt.gca().set_aspect("equal")

cb = plt.colorbar(scatter, aspect=30)
cb.set_label('log(age/yr)', fontsize=14)

plt.xlabel('$\\Delta$RA (arcsec)', fontsize=14)
plt.xlim(plt.xlim()[::-1]) # reverse
plt.ylabel('$\\Delta$Dec (arcsec)', fontsize=14)
plt.grid(True, linestyle='-', alpha=0.1)

print(plt.xlim(), plt.ylim())

for spine in plt.gca().spines.values():
    spine.set_visible(False)

plt.text(0.03, 0.97, "Star Cluster\nGraph", transform=plt.gca().transAxes, va="top", fontsize=16)

plt.gca().set_rasterization_zorder(3) # Rasterize everything with zorder < 3

plt.savefig(results_dir / "figures/NGC1566-graph.pdf", dpi=300, bbox_inches="tight")

In [ ]:
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.visualization import make_lupton_rgb
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import numpy as np


fig = plt.figure(figsize=(11, 4.5), dpi=300, constrained_layout=True)

gs = GridSpec(1, 3, figure=fig, width_ratios=[1, 1, 0.025], wspace=0.025, hspace=0)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
cax = fig.add_subplot(gs[2])


# right panel first (since that dictates the x/ylims)
selection = datum.edge_attr[:, 0] < 15
edge_index = datum.edge_index[:, selection].T.numpy()

node_pos = datum.pos.numpy()
node_colors = datum.y.numpy()


lc = LineCollection(node_pos[edge_index], colors='k', alpha=0.2, linewidths=0.2, zorder=0, rasterized=True)
ax2.add_collection(lc)

cmap = cmr.get_sub_cmap(cmr.torch, 0.2, 0.9, N=8)

scatter = ax2.scatter(
    node_pos[:, 0],
    node_pos[:, 1],
    c=node_colors,
    s=10,
    cmap=cmap,
    vmin=6,
    vmax=10,
    edgecolors='none',
    zorder=2
)

ax2.set_xlabel('$\\Delta$RA (arcsec)', fontsize=14)
ax2.set_xlim(ax2.get_xlim()[::-1]) # reverse
ax2.set_ylabel('$\\Delta$Dec (arcsec)', fontsize=14)
ax2.grid(alpha=0.15)

# For gnomonic projection in second panel
xlim = ax2.get_xlim()
ylim = ax2.get_ylim()

# left panel

data_dir = f"{base_dir}/data/ngc1566-images"
r_data = fits.getdata(f"{data_dir}/hlsp_phangs-hst_hst_wfc3-uvis_ngc1566_f814w_v1_exp-drc-sci.fits")
g_data = fits.getdata(f"{data_dir}/hlsp_phangs-hst_hst_wfc3-uvis_ngc1566_f555w_v1_exp-drc-sci.fits")
b_data = fits.getdata(f"{data_dir}/hlsp_phangs-hst_hst_wfc3-uvis_ngc1566_f438w_v1_exp-drc-sci.fits")

from astropy.stats import sigma_clipped_stats
def preprocess(data):
    mean, median, std = sigma_clipped_stats(data, sigma=3.0)
    return np.maximum(data - median, 0) 
r_data = preprocess(r_data) * 1.0
g_data = preprocess(g_data) * 1.5
b_data = preprocess(b_data) * 5.0

# Get WCS from one of the FITS files
wcs = WCS(fits.getheader(f"{data_dir}/hlsp_phangs-hst_hst_wfc3-uvis_ngc1566_f814w_v1_exp-drc-sci.fits"))

pixel_scales = proj_plane_pixel_scales(wcs) * 3600 
pixel_scale = pixel_scales[0]

center_ra, center_dec = datum.center.numpy()

# Convert center RA/DEC to pixel coordinates
center_px, center_py = wcs.world_to_pixel_values(center_ra, center_dec)

# Define cutout size (in arcsec, matching the graph extent)
arcsec_extent = max(abs(xlim[0] - xlim[1]), abs(ylim[0] - ylim[1])) / 2

cutout_size = int(arcsec_extent / pixel_scale)

# Extract cutout around center
y1, y2 = int(center_py - cutout_size), int(center_py + cutout_size)
x1, x2 = int(center_px - cutout_size), int(center_px + cutout_size)
r_cutout = r_data[y1:y2, x1:x2]
g_cutout = g_data[y1:y2, x1:x2]
b_cutout = b_data[y1:y2, x1:x2]


rgb = make_lupton_rgb(r_cutout, g_cutout, b_cutout, Q=8, stretch=0.15, minimum=0)

extent_arcsec = [
    (x2 - center_px) * pixel_scale,  
    (x1 - center_px) * pixel_scale,  
    (y1 - center_py) * pixel_scale,  
    (y2 - center_py) * pixel_scale   
]

ax1.imshow(rgb, origin='lower', extent=extent_arcsec)
ax1.set_xlabel('$\\Delta$RA (arcsec)', fontsize=14)
ax1.set_ylabel('$\\Delta$Dec (arcsec)', fontsize=14)
ax1.set_aspect("equal")

ax1.grid(alpha=0.15, c='w')

ax1.text(0.03, 0.97, "HST WFC3", color="white", transform=ax1.transAxes, va="top", fontsize=16)
ax1.text(0.03, 0.91, "F814W", color="red", transform=ax1.transAxes, va="top", fontsize=16)
ax1.text(0.03, 0.85, "F555W", color="green", transform=ax1.transAxes, va="top", fontsize=16)
ax1.text(0.03, 0.79, "F438W", color="blue", transform=ax1.transAxes, va="top", fontsize=16)

# keeping things aligned...?
ax1.set_xlim(xlim)
ax1.set_ylim(ylim)
ax2.set_xlim(xlim)
ax2.set_ylim(ylim)

ax1.set_aspect('equal', adjustable='box')
ax2.set_aspect('equal', adjustable='box')
cb = fig.colorbar(scatter, cax=cax, aspect=20)
cb.set_label('log(age/yr)', fontsize=14)

gs.update(wspace=0.025)

pos1 = ax1.get_position()
pos2 = ax2.get_position()
ax1.set_position([pos1.x0, pos1.y0, pos1.width * 0.95, pos1.height])
ax2.set_position([pos2.x0 - 0.01, pos2.y0, pos2.width * 0.95, pos2.height])
cax.set_position([cax.get_position().x0- 0.025, pos2.y0, cax.get_position().width, pos2.height])

# no spines
for ax in [ax1, ax2]:
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.savefig(results_dir / "figures/NGC1566-rgb-graph.pdf", dpi=300, bbox_inches="tight")

# Pred vs True

In [ ]:
df_gnn = pd.concat([pd.read_csv(results_dir / f"gnn/cv_gnn_fold_{k}_predictions.csv") for k in range(5)])
df_rf = pd.concat([pd.read_csv(results_dir / f"rf/cv_rf_fold_{k}_predictions.csv") for k in range(5)])

In [ ]:
assert len(df_gnn) == len(df_rf)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.5, 4), dpi=300, sharex=False, sharey=True)

colors = [c0, c3]
lims = (5.8, 10.3)
labels = ["Photometry Only (RF)", "Photometry + Environment (GNN)"]

for model, df, ax, color in zip(labels, [df_rf, df_gnn], axes.flat, colors):
    is_finite = np.isfinite(df["y_true"])
    p = df[f"y_pred"][is_finite]
    y = df[f"y_true"][is_finite]

    cmap = LinearSegmentedColormap.from_list("my_cmap", ["#ffffff", color])
    
    # ax.scatter(y, p, s=0.3, alpha=1, color=color, linewidths=0, edgecolors="none", rasterized=True)
    ax.hist2d(y, p, bins=[42, 42], range=[(6, 10.2), (6, 10.2)], norm=LogNorm(vmin=1, vmax=3e2), cmap=cmap, rasterized=True)

    ax.plot(lims, lims, ls='-', c='w', lw=1.5)
    ax.plot(lims, lims, ls='-', c='0.5', lw=0.3)
    
    ax.set_xlim(*lims)
    ax.set_ylim(*lims)
    ax.set_xticks(range(*(map(lambda x: np.ceil(x).astype(int), lims))))
    ax.set_yticks(range(*(map(lambda x: np.ceil(x).astype(int), lims))))

    # ax.text(0.03, 0.9, model, fontsize=16, ha="left", transform=ax.transAxes)
    ax.set_title(model, fontsize=12, color=color, weight="bold")

    ax.set_aspect("equal")
    ax.grid(alpha=0.15)
    
    for z, [metric, func] in enumerate(metrics_mapping.items()):
        if metric in ["Outlier Frac.", "Bias", "$R^2$"]: continue
        score = func(p, y)
        ax.text(0.95, 0.06*(-0.2 + z), f"{metric: >7s} = {score:.3f}", fontsize=12, ha="right", transform=ax.transAxes)
    
        
fig.subplots_adjust(wspace=0.025, hspace=0.025, left=0.075, right=0.975, top=0.975, bottom=0.075)

axes[0].set_ylabel(r"Predicted log(age/yr)", fontsize=14)

axes[0].set_xlabel(r"True log(age/yr)", fontsize=14)
axes[1].set_xlabel(r"True log(age/yr)", fontsize=14)

fig.tight_layout()
# plt.savefig(results_dir / "figures/results_pred-vs-true.pdf")

# plt.clf()

# CDF plots

In [ ]:
absolute_errors_gnn = np.sort((df_gnn.y_true - df_gnn.y_pred).abs())
absolute_errors_rf = np.sort((df_rf.y_true - df_rf.y_pred).abs())

y_steps = np.arange(1, len(absolute_errors_gnn) + 1) / len(absolute_errors_gnn)

fig, ax = plt.subplots(1,1,figsize=(4,4), dpi=300)

ax.plot(absolute_errors_gnn, y_steps, drawstyle='steps-post', c=c3, lw=3)
ax.plot(absolute_errors_rf, y_steps, drawstyle='steps-post', c=c0, lw=3)

ax.set_xscale("log")
ax.set_xlim(1e-2, 1)
ax.set_ylim(0, 1)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
ax.set_xticks([0.01, 0.03, 0.1, 0.3, 1], [0.01, 0.03, 0.1, 0.3, 1])

ax.text(0.105, 0.55, "GNN", fontsize=14, fontweight="bold", color=c3, path_effects=[pe.withStroke(linewidth=7, foreground="white")], rotation=50)
ax.text(0.185, 0.58, "RF", fontsize=14, fontweight="bold", color=c0, path_effects=[pe.withStroke(linewidth=7, foreground="white")], rotation=50)

ax.set_xlabel("Absolute Error (dex)", fontsize=14)
ax.set_ylabel("CDF", fontsize=14)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(axis="x", which="major", labelsize=10)

ax.grid(alpha=0.15)

# plt.savefig(results_dir / "figures/cdf-errors.pdf", bbox_inches="tight")

In [ ]:
for percentile in [0.5, 0.75, 0.9, 0.95, 0.99]:
    aegnn = absolute_errors_gnn[np.argmin((y_steps - percentile)**2)]
    aerf = absolute_errors_rf[np.argmin((y_steps - percentile)**2)]

    print(f"{percentile} | {aegnn:.4f} | {aerf:.4f}")

In [ ]:
df_gnn.shape, df_gnn.y_true.notna().sum()

# Correlations betwee errors and galaxy properties

In [ ]:
with open(results_dir / "cv_galaxy_splits.json", 'r') as f:
    cv_splits = json.load(f)

In [ ]:
galaxy_ordering = list(itertools.chain(*(s['valid'] for s in cv_splits)))

In [ ]:

sample_metadata = Table.read(base_dir / "data/Leroy+2021_table3.fits").to_pandas()
sample_metadata["Galaxy"] = [s.decode('utf-8').strip() for s in sample_metadata["Name"]]
sample_metadata["Galaxy"] = sample_metadata["Galaxy"].str.replace("NGC", "NGC_").str.replace("IC", "IC_")
sample_metadata = sample_metadata.set_index("Galaxy").rename({"NGC_685": "NGC_0685"})

galaxies_meta = sample_metadata.loc[galaxy_ordering]

In [ ]:
results = {}
num_nodes = []
start = 0
for i, g in enumerate(galaxy_ordering):
    end = start + len(data_dict[g].y)
    
    results[g] = dict()
    results[g]["graph"] = data_dict[g]
    num_nodes.append(data_dict[g].num_nodes)
    
    results[g]["log_ages"] = df_gnn["y_true"].values[start:end]
    results[g]["gnn_pred"] = df_gnn["y_pred"].values[start:end]
    results[g]["rf_pred"] = df_rf["y_pred"].values[start:end]
    
    results[g]["gnn_metrics"] = {metric: func(results[g]["gnn_pred"], results[g]["log_ages"]) for metric, func in metrics_mapping.items()}
    results[g]["rf_metrics"] = {metric: func(results[g]["rf_pred"], results[g]["log_ages"]) for metric, func in metrics_mapping.items()}
    
    results[g]["metadata"] = galaxies_meta.loc[g]

    start = end

In [ ]:
for error_metric in ["RMSE", "MAE", "NMAD"]:
    
    dep_vars = {
        r"Num. clusters": [len(results[g]["log_ages"]) for g in galaxy_ordering],
        r"Frac. old": [np.mean(results[g]["log_ages"] > 10) for g in galaxy_ordering], 
        r"Frac. young": [np.mean(results[g]["log_ages"] <= 7) for g in galaxy_ordering], 
        r"Effective radius": [results[g]["metadata"]["Re"] for g in galaxy_ordering], 
        r"Distance": [results[g]["metadata"]["Dist"] for g in galaxy_ordering],   
        r"Inclination angle": [results[g]["metadata"]["i"] for g in galaxy_ordering], 
        # r"Position angle": [results[g]["metadata"]["PA"] for g in galaxy_ordering], 
        r"SFR": [results[g]["metadata"]["logSFR"] for g in galaxy_ordering], 
        r"CO luminosity": [results[g]["metadata"]["logLCO"] for g in galaxy_ordering], 
        r"HI mass": [results[g]["metadata"]["logMHI"] for g in galaxy_ordering], 
    }
    
    errors = [results[g]["gnn_metrics"][error_metric] for g in galaxy_ordering]
    rf_errors = [results[g]["rf_metrics"][error_metric] for g in galaxy_ordering]
    
    fig, axes = plt.subplots(3, 3, figsize=(9, 9), dpi=150, sharey=True)
    for i, [[varname, vars], ax] in enumerate(zip(dep_vars.items(), axes.flat)):
    
        ax.scatter(vars, errors, c=c3) # color=cmap(i / (len(dep_vars) - 1)))
        ax.scatter(vars, rf_errors, c=c0)

        ax.text(
            0.025, 0.85, 
            f"${scipy.stats.spearmanr(vars, errors, nan_policy='omit').statistic:+.2f}$" + (" (GNN)" if i == 0 else ""), 
            color=c3, fontsize=14, transform=ax.transAxes, ha="left", va="top"
        )
        ax.text(
            0.025, 0.95, 
            f"${scipy.stats.spearmanr(vars, rf_errors, nan_policy='omit').statistic:+.2f}$" + (" (RF)" if i == 0 else ""), 
            color=c0, fontsize=14, transform=ax.transAxes, ha="left", va="top"
        )

        # if i == 0:
        #     ax.text(0.02, 1.0, "Spearman $r$", color='k', fontsize=14, transform=ax.transAxes, ha="left", va="top")
    
        if i % 3 == 0:
            ax.set_ylabel(error_metric + " [dex]")
        ax.set_xlabel(varname)
        ax.grid(alpha=0.05)
        
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", which="major", labelsize=10)
    
    fig.tight_layout()
    
    plt.savefig(results_dir / f"figures/all_error_correlations-{error_metric}.pdf", dpi=300)

In [ ]:
# error_metric = "MAE"
# dep_vars = {
#     r"Num. clusters": [len(results[g]["log_ages"]) for g in galaxy_ordering],
#     r"Frac. old": [np.mean(results[g]["log_ages"] > 10) for g in galaxy_ordering], 
#     r"Frac. young": [np.mean(results[g]["log_ages"] <= 7) for g in galaxy_ordering], 
#     r"Effective radius": [results[g]["metadata"]["Re"] for g in galaxy_ordering], 
#     r"Distance": [results[g]["metadata"]["Dist"] for g in galaxy_ordering],   
#     r"Inclination angle": [results[g]["metadata"]["i"] for g in galaxy_ordering], 
#     # r"Position angle": [results[g]["metadata"]["PA"] for g in galaxy_ordering], 
#     r"SFR": [results[g]["metadata"]["logSFR"] for g in galaxy_ordering], 
#     r"CO luminosity": [results[g]["metadata"]["logLCO"] for g in galaxy_ordering], 
#     r"HI mass": [results[g]["metadata"]["logMHI"] for g in galaxy_ordering], 
# }


# errors = [results[g]["gnn_metrics"][error_metric] for g in galaxy_ordering]
# rf_errors = [results[g]["rf_metrics"][error_metric] for g in galaxy_ordering]

# fig, axes = plt.subplots(3, 3, figsize=(8, 9), dpi=150, sharey=True)
# for i, [[varname, vars], ax] in enumerate(zip(dep_vars.items(), axes.flat)):
#     ax.scatter(vars, errors, c=c3) # color=cmap(i / (len(dep_vars) - 1)))
#     ax.scatter(vars, rf_errors, c=c0)
#     ax.text(0.08, 0.85, f"${scipy.stats.spearmanr(vars, errors, nan_policy='omit').statistic:+.2f}$", color=c3, fontsize=14, transform=ax.transAxes, ha="left", va="top")
#     ax.text(0.08, 0.95, f"${scipy.stats.spearmanr(vars, rf_errors, nan_policy='omit').statistic:+.2f}$", color=c0, fontsize=14, transform=ax.transAxes, ha="left", va="top")
#     ax.set_title(varname, fontsize=14)
#     ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
    
#     ax.grid(alpha=0.15)

# axes[0][0].set_ylabel(f"{error_metric}", fontsize=14)
# axes[1][0].set_ylabel(f"{error_metric}", fontsize=14)
# axes[2][0].set_ylabel(f"{error_metric}", fontsize=14)
# fig.tight_layout()

# plt.savefig(results_dir / f"figures/all_error_correlations-{error_metric}.pdf", dpi=300)

# Errors in different age regimes

In [ ]:
q = Query(QueryMaker.isfinite("y_true"))

for z, [metric, func] in enumerate(metrics_mapping.items()):
    score = func(q.filter(df_gnn).y_pred, q.filter(df_gnn).y_true)
    print(metric, score)

for z, [metric, func] in enumerate(metrics_mapping.items()):
    score = func(q.filter(df_rf).y_pred, q.filter(df_rf).y_true)
    print(metric, score)

In [ ]:
q = Query(QueryMaker.isfinite("y_true"), "y_true <= 7")

for z, [metric, func] in enumerate(metrics_mapping.items()):
    score = func(q.filter(df_gnn).y_pred, q.filter(df_gnn).y_true)
    print(metric, score)


In [ ]:
q = Query(QueryMaker.isfinite("y_true"), "y_true > 7", "y_true < 9")

for z, [metric, func] in enumerate(metrics_mapping.items()):
    score = func(q.filter(df_gnn).y_pred, q.filter(df_gnn).y_true)
    print(metric, score)


In [ ]:
q = Query(QueryMaker.isfinite("y_true"), "y_true > 9")

for z, [metric, func] in enumerate(metrics_mapping.items()):
    score = func(q.filter(df_gnn).y_pred, q.filter(df_gnn).y_true)
    print(metric, score)


In [ ]:
q = Query(QueryMaker.isfinite("y_true"), "y_true <= 7")

for z, [metric, func] in enumerate(metrics_mapping.items()):
    score = func(q.filter(df_rf).y_pred, q.filter(df_rf).y_true)
    print(metric, score)


In [ ]:
# error_metric = "MAE"

qs = {
    "< 10 Myr": Query(QueryMaker.isfinite("y_true"), "y_true <= 7"),
    "10 Myr to\n1 Gyr": Query(QueryMaker.isfinite("y_true"), "y_true > 7", "y_true < 9"),
    "> 1 Gyr": Query(QueryMaker.isfinite("y_true"), "y_true > 9")
}

scatters = [0.3, 0.1, 0.4]

figure, axes = plt.subplots(3, 3, figsize=(9,3), dpi=150, sharey=True, sharex=True)

for ax_row, [label, q], scatter in zip(axes, qs.items(), scatters):
    for error_metric, ax in zip(["NMAD", "MAE", "RMSE"], ax_row):
        f = metrics_mapping[error_metric]

        # ax.barh(
        #     0,
        #     scatter,
        #     color=c4,
        # )

        ax.axvline(scatter, c=c4, lw=2)
        # ax.axvline(scatter, c="white", lw=0.5)
        
        ax.barh(
            1,
            f(q.filter(df_gnn).y_pred, q.filter(df_gnn).y_true),
            color=c3
        )
        
        ax.barh(
            2,
            f(q.filter(df_rf).y_pred, q.filter(df_rf).y_true),
            color=c0,
        )
        ax.grid(alpha=0.15, axis='x')
        ax.set_xlim(0, 0.6)
        
        # ax.set_yticks([0, 1, 2], ["Uncertainty", "GNN", "RF"], rotation=60)
        ax.set_yticks([], [])
    
        # ax.grid(alpha=0.05)
        
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        
        ax.tick_params(axis="x", which="major", labelsize=10)

        if error_metric == "NMAD":
            ax.text(-0.05, 0.5, label, fontsize=12, ha="right", va="center", transform=ax.transAxes)

        if label == "> 1 Gyr":
            ax.set_xlabel(f"{error_metric} [dex]", fontsize=12)

axes[0,0].text(0.01, 0.98, "GNN", color="white", va="center", fontweight="bold", fontsize=14)
axes[0,0].text(0.01, 1.98, "RF", color="white", va="center", fontweight="bold", fontsize=14)
    
# plt.savefig(results_dir / "figures/lower-bounds-on-errors.pdf", bbox_inches="tight")

# Residuals

In [ ]:
for galname in results.keys():
    res = results[galname]
    
    node_pos = res["graph"].pos
    edge_list = res["graph"].edge_index
    
    log_ages = res["log_ages"]
    gnn_preds = res["gnn_pred"]
    rf_preds = res["rf_pred"]
    
    scatter_kws = dict(
        s=25,
        cmap=cmr.get_sub_cmap(cmr.fusion, 0.25, 0.75),
        vmin=-0.8,
        vmax=0.8,
        edgecolors='k',
        lw=0.1,
        zorder=2,
        rasterized=True
    )
    
    fig = plt.figure(figsize=(10, 4.5), dpi=300)
    
    subfigs = fig.subfigures(1, 2, width_ratios=[1, 0.1], wspace=-0.14)
    
    axs_left = subfigs[0].subplots(1, 2, sharey=True)
    subfigs[0].subplots_adjust(wspace=0.)
    
    ax1 = axs_left[0]
    scatter = ax1.scatter(node_pos[:, 0], node_pos[:, 1], c=gnn_preds - log_ages, **scatter_kws)
    ax1.set_aspect("equal")
    ax1.set_xlabel('$\\Delta$RA (arcsec)', fontsize=14)
    ax1.set_xlim(ax1.get_xlim()[::-1])
    ax1.set_ylabel('$\\Delta$Dec (arcsec)', fontsize=14)
    ax1.grid(True, linestyle='-', alpha=0.1)
    ax1.set_title(galname.replace("_", " ") + " GNN Residuals", fontsize=16)
    
    
    ax2 = axs_left[1]
    ax2.scatter(node_pos[:, 0], node_pos[:, 1], c=rf_preds - log_ages, **scatter_kws)
    ax2.set_aspect("equal")
    ax2.set_xlabel('$\\Delta$RA (arcsec)', fontsize=14)
    ax2.set_xlim(ax2.get_xlim()[::-1])
    ax2.grid(True, linestyle='-', alpha=0.1)
    ax2.set_title(galname.replace("_", " ") + " RF Residuals", fontsize=16)
    
    
    cax = subfigs[1].add_axes([0, 0.1, 0.2, 0.78]) # [left, bottom, width, height]
    cb = fig.colorbar(scatter, cax=cax)
    cb.set_label('Residual log(age/yr)', fontsize=14)
    
    plt.savefig(results_dir / f"figures/graphs/{galname}-residuals.pdf", dpi=300, bbox_inches='tight')
    
    plt.show()

# Extreme errors

In [ ]:
scatter_kws = dict(
    s=25,
    cmap=cmr.get_sub_cmap(cmr.fusion, 0.25, 0.75),
    vmin=-0.8,
    vmax=0.8,
    edgecolors='k',
    lw=0.1,
    zorder=2,
    rasterized=True
)

for galname in results.keys():
    # print(galname)
    res = results[galname]
    node_pos = res["graph"].pos

    log_ages = res["log_ages"]
    gnn_preds = res["gnn_pred"]
    rf_preds = res["rf_pred"]

    # very high errors positive errors: ground truth is < 10 Myr and GNN pred is > 500 Myr
    overestimates = (log_ages < 7) & (gnn_preds > 8.7)

    radec = (node_pos.numpy() / 3600 + results[galname]["graph"].center.numpy())
    # print(radec[overestimates])
    # print((gnn_preds - log_ages)[overestimates])
    
    if sum(overestimates) > 0:
        fig = plt.figure(figsize=(4, 4), dpi=300)
        plt.scatter(node_pos[:, 0], node_pos[:, 1], c=gnn_preds - log_ages, **scatter_kws)
        plt.scatter(node_pos[overestimates][:, 0], node_pos[overestimates][:, 1], c='k', marker='x', zorder=9, alpha=1)
        plt.title(galname)
        plt.xlim(plt.xlim()[::-1])
        plt.gca().set_aspect("equal")

        for ra_deg, dec_deg in radec[overestimates]:
            coord = SkyCoord(ra=ra_deg*u.degree, dec=dec_deg*u.degree, frame='icrs')
            ra_hms = coord.ra.to_string(unit=u.hour, sep=':')
            dec_dms = coord.dec.to_string(unit=u.deg, sep=':')
        
            # print(ra_deg, dec_deg, ra_hms, dec_dms)

## actual cutouts

In [ ]:
gal_image_name_mappings = {
    "NGC_5248": "ngc5248",
    "NGC_1097": "ngc1097mosaic",
    "NGC_4536": "ngc4536mosaic",
    "NGC_4535": "ngc4535",
    "NGC_4321": "ngc4321mosaic",
    "NGC_3627": "ngc3627mosaic",
    "NGC_4254": "ngc4254mosaic",
    "NGC_4826": "ngc4826",
    "NGC_1385": "ngc1385",
    "NGC_1365": "ngc1365",
    "NGC_4298": "ngc4298",
    "NGC_2835": "ngc2835",
    "NGC_2903": "ngc2903mosaic",
    "NGC_3351": "ngc3351mosaic",
}

overestimate_color_images = []

for galname in results.keys():
    # print(galname)
    res = results[galname]
    node_pos = res["graph"].pos

    log_ages = res["log_ages"]
    gnn_preds = res["gnn_pred"]
    rf_preds = res["rf_pred"]

    # very high errors positive errors: ground truth is < 10 Myr and GNN pred is > 500 Myr
    overestimates = (log_ages < 7) & (gnn_preds > 8.7)

    origin = res["graph"].center
    radec = np.vstack([
        node_pos[:,0] / 3600 / np.cos(np.deg2rad(origin[1])) + origin[0],
        node_pos[:,1] / 3600 + origin[1]
    ]).T
    # print(radec[overestimates])
    # print((gnn_preds - log_ages)[overestimates])
    
    if sum(overestimates) > 0:
        
        gal_image_name = gal_image_name_mappings[galname] # based on naming scheme from https://archive.stsci.edu/hlsps/phangs-hst/
        
        for ra_deg, dec_deg in tqdm.tqdm(radec[overestimates]):
            coord = SkyCoord(ra=ra_deg*u.degree, dec=dec_deg*u.degree, frame='icrs')
            ra_hms = coord.ra.to_string(unit=u.hour, sep=':')
            dec_dms = coord.dec.to_string(unit=u.deg, sep=':')

            
            input_files = [
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f814w_v1_exp-drc-sci.fits",
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f555w_v1_exp-drc-sci.fits",
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f438w_v1_exp-drc-sci.fits",
            ]
            
            center_coord = SkyCoord(f"{ra_deg} {dec_deg}", unit="deg")
            cutout_size = [64, 64]
            
            color_image = FITSCutout(input_files, center_coord, cutout_size).get_image_cutouts(colorize=True)[0]
            overestimate_color_images.append(color_image)

## Examples of good predictions from same host galaxies

In [ ]:
gal_image_name_mappings = {
    "NGC_5248": "ngc5248",
    "NGC_1097": "ngc1097mosaic",
    "NGC_4536": "ngc4536mosaic",
    "NGC_4535": "ngc4535",
    "NGC_4321": "ngc4321mosaic",
    "NGC_3627": "ngc3627mosaic",
    "NGC_4254": "ngc4254mosaic",
    "NGC_4826": "ngc4826",
    "NGC_1385": "ngc1385",
    "NGC_1365": "ngc1365",
    "NGC_4298": "ngc4298",
    "NGC_2835": "ngc2835",
    "NGC_2903": "ngc2903mosaic",
    "NGC_3351": "ngc3351mosaic",
}

accurate_young_color_images = []

for galname in results.keys():
    # print(galname)
    res = results[galname]
    node_pos = res["graph"].pos

    log_ages = res["log_ages"]
    gnn_preds = res["gnn_pred"]
    rf_preds = res["rf_pred"]

    accurate_young = ((gnn_preds - log_ages)**2 < 0.1**2) & (log_ages < 7.5)

    origin = res["graph"].center
    radec = np.vstack([
        node_pos[:,0] / 3600 / np.cos(np.deg2rad(origin[1])) + origin[0],
        node_pos[:,1] / 3600 + origin[1]
    ]).T
    # print(radec[overestimates])
    # print((gnn_preds - log_ages)[overestimates])
    
    if sum(accurate_young) > 0 and galname in gal_image_name_mappings.keys():
        
        gal_image_name = gal_image_name_mappings[galname] # based on naming scheme from https://archive.stsci.edu/hlsps/phangs-hst/

        rng = np.random.RandomState(42)
        random_indices = rng.choice(radec[accurate_young].shape[0], size=4, replace=False)
        for ra_deg, dec_deg in tqdm.tqdm(radec[accurate_young][random_indices]):
            coord = SkyCoord(ra=ra_deg*u.degree, dec=dec_deg*u.degree, frame='icrs')
            ra_hms = coord.ra.to_string(unit=u.hour, sep=':')
            dec_dms = coord.dec.to_string(unit=u.deg, sep=':')

            
            input_files = [
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f814w_v1_exp-drc-sci.fits",
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f555w_v1_exp-drc-sci.fits",
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f438w_v1_exp-drc-sci.fits",
            ]
            
            center_coord = SkyCoord(f"{ra_deg} {dec_deg}", unit="deg")
            cutout_size = [64, 64]
            
            color_image = FITSCutout(input_files, center_coord, cutout_size).get_image_cutouts(colorize=True)[0]
            accurate_young_color_images.append(color_image)

In [ ]:
gal_image_name_mappings = {
    "NGC_5248": "ngc5248",
    "NGC_1097": "ngc1097mosaic",
    "NGC_4536": "ngc4536mosaic",
    "NGC_4535": "ngc4535",
    "NGC_4321": "ngc4321mosaic",
    "NGC_3627": "ngc3627mosaic",
    "NGC_4254": "ngc4254mosaic",
    "NGC_4826": "ngc4826",
    "NGC_1385": "ngc1385",
    "NGC_1365": "ngc1365",
    "NGC_4298": "ngc4298",
    "NGC_2835": "ngc2835",
    "NGC_2903": "ngc2903mosaic",
    "NGC_3351": "ngc3351mosaic",
}

accurate_old_color_images = []

for galname in results.keys():
    # print(galname)
    res = results[galname]
    node_pos = res["graph"].pos

    log_ages = res["log_ages"]
    gnn_preds = res["gnn_pred"]
    rf_preds = res["rf_pred"]

    accurate_old = ((gnn_preds - log_ages)**2 < 0.1**2) & (log_ages > 9.0)

    origin = res["graph"].center
    radec = np.vstack([
        node_pos[:,0] / 3600 / np.cos(np.deg2rad(origin[1])) + origin[0],
        node_pos[:,1] / 3600 + origin[1]
    ]).T
    # print(radec[overestimates])
    # print((gnn_preds - log_ages)[overestimates])
    
    if sum(accurate_old) > 0 and galname in gal_image_name_mappings.keys():
        
        gal_image_name = gal_image_name_mappings[galname] # based on naming scheme from https://archive.stsci.edu/hlsps/phangs-hst/

        rng = np.random.RandomState(42)
        random_indices = rng.choice(radec[accurate_old].shape[0], size=4, replace=False)
        for ra_deg, dec_deg in tqdm.tqdm(radec[accurate_old][random_indices]):
            coord = SkyCoord(ra=ra_deg*u.degree, dec=dec_deg*u.degree, frame='icrs')
            ra_hms = coord.ra.to_string(unit=u.hour, sep=':')
            dec_dms = coord.dec.to_string(unit=u.deg, sep=':')

            
            input_files = [
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f814w_v1_exp-drc-sci.fits",
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f555w_v1_exp-drc-sci.fits",
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f438w_v1_exp-drc-sci.fits",
            ]
            
            center_coord = SkyCoord(f"{ra_deg} {dec_deg}", unit="deg")
            cutout_size = [64, 64]
            
            color_image = FITSCutout(input_files, center_coord, cutout_size).get_image_cutouts(colorize=True)[0]
            accurate_old_color_images.append(color_image)

In [ ]:
gal_image_name_mappings = {
    "NGC_5248": "ngc5248",
    "NGC_1097": "ngc1097mosaic",
    "NGC_4536": "ngc4536mosaic",
    "NGC_4535": "ngc4535",
    "NGC_4321": "ngc4321mosaic",
    "NGC_3627": "ngc3627mosaic",
    "NGC_4254": "ngc4254mosaic",
    "NGC_4826": "ngc4826",
    "NGC_1385": "ngc1385",
    "NGC_1365": "ngc1365",
    "NGC_4298": "ngc4298",
    "NGC_2835": "ngc2835",
    "NGC_2903": "ngc2903mosaic",
    "NGC_3351": "ngc3351mosaic",
}

underprediction_color_images = []

for galname in results.keys():
    # print(galname)
    res = results[galname]
    node_pos = res["graph"].pos

    log_ages = res["log_ages"]
    gnn_preds = res["gnn_pred"]
    rf_preds = res["rf_pred"]

    underpredictions = (gnn_preds < 8.) & (log_ages >= 9.5)
    origin = res["graph"].center
    radec = np.vstack([
        node_pos[:,0] / 3600 / np.cos(np.deg2rad(origin[1])) + origin[0],
        node_pos[:,1] / 3600 + origin[1]
    ]).T
    # print(radec[overestimates])
    # print((gnn_preds - log_ages)[overestimates])
    
    if sum(underpredictions) > 0 and galname in gal_image_name_mappings.keys():
        
        gal_image_name = gal_image_name_mappings[galname] # based on naming scheme from https://archive.stsci.edu/hlsps/phangs-hst/
        
        for ra_deg, dec_deg in tqdm.tqdm(radec[underpredictions]):
            coord = SkyCoord(ra=ra_deg*u.degree, dec=dec_deg*u.degree, frame='icrs')
            ra_hms = coord.ra.to_string(unit=u.hour, sep=':')
            dec_dms = coord.dec.to_string(unit=u.deg, sep=':')

            
            input_files = [
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f814w_v1_exp-drc-sci.fits",
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f555w_v1_exp-drc-sci.fits",
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f438w_v1_exp-drc-sci.fits",
            ]
            
            center_coord = SkyCoord(f"{ra_deg} {dec_deg}", unit="deg")
            cutout_size = [64, 64]
            
            color_image = FITSCutout(input_files, center_coord, cutout_size).get_image_cutouts(colorize=True)[0]
            underprediction_color_images.append(color_image)

## Show em all


In [ ]:
len(accurate_young_color_images), len(underprediction_color_images)

In [ ]:
random.seed(42)

im_overestimates = random.sample(overestimate_color_images, k=12)
im_accurate_young = random.sample(accurate_young_color_images, k=12)
im_underestimates = random.sample(underprediction_color_images, k=12)
im_accurate_old = random.sample(accurate_old_color_images, k=12)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec

# colorblind friendly
CB_GREEN = '#009E73'
CB_RED = '#D55E00'

fig = plt.figure(figsize=(20, 15))

# make a 2x2 "supergrid"
outer_grid = gridspec.GridSpec(2, 2, wspace=0.08, hspace=0.12)

def fill_block(outer_pos, imgs, color):
    # make a nested 3x4 grid inside the supergrid cell
    inner_grid = gridspec.GridSpecFromSubplotSpec(3, 4, subplot_spec=outer_pos, wspace=0.03, hspace=0.03)
    
    # bounding box for supergrid rectangle
    all_axes = []
    for i, img in enumerate(imgs):
        ax = plt.Subplot(fig, inner_grid[i])
        ax.imshow(img)
        ax.axis('off')
        fig.add_subplot(ax)
        all_axes.append(ax)
    
    # draw rectangle
    inv = fig.transFigure.inverted()
    
    # draw rectangles around bounding boxes
    pos0 = all_axes[0].get_position()
    posN = all_axes[-1].get_position()
    
    rect = patches.Rectangle(
        (pos0.x0 - 0.006, posN.y0 - 0.008), 
        (posN.x1 - pos0.x0 + 0.012), 
        (pos0.y1 - posN.y0 + 0.016),
        linewidth=6, edgecolor=color, facecolor='none', 
        transform=fig.transFigure, clip_on=False
    )
    fig.add_artist(rect)

# Fill the 4 blocks
fill_block(outer_grid[0, 0], im_overestimates, CB_RED) # Top-Left
fill_block(outer_grid[0, 1], im_accurate_old,  CB_GREEN)   # Top-Right
fill_block(outer_grid[1, 0], im_accurate_young, CB_GREEN)   # Bottom-Left
fill_block(outer_grid[1, 1], im_underestimates, CB_RED) # Bottom-Right

# Add Supergrid Labels
# Adjusted coordinates slightly to account for the new gaps
fig.text(0.32, 0.08, 'Target: Young', ha='center', va='center', fontsize=20, fontweight='bold')
fig.text(0.72, 0.08, 'Target: Old', ha='center', va='center', fontsize=20, fontweight='bold')
fig.text(0.1, 0.72, 'GNN: Old', ha='center', va='center', rotation='vertical', fontsize=20, fontweight='bold')
fig.text(0.1, 0.3, 'GNN: Young', ha='center', va='center', rotation='vertical', fontsize=20, fontweight='bold')

plt.savefig(results_dir / "figures/cutout-grid.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(4, 12, figsize=(12.9, 4), dpi=300, )

for ims, ax_row in zip([im_overestimates, im_accurate_young, im_underestimates, im_accurate_old], axes):
    for im, ax in zip(ims, ax_row):
        ax.imshow(im)
        ax.axis("off")

fig.text(0, 0.86, "GNN: old\nTrue: young", fontsize=12, va="center")
fig.text(0, 0.625, "GNN: young\nTrue: young", fontsize=12, va="center")
fig.text(0, 0.375, "GNN: young\nTrue: old", fontsize=12, va="center")
fig.text(0, 0.14, "GNN: old\nTrue: old", fontsize=12, va="center")

# fig.subplots_adjust(left=0.09, wspace=0.02, hspace=0.02, top=0.98, bottom=0.02, right=0.98)
# plt.savefig(results_dir / "figures/poor-predictions.pdf", bbox_inches="tight")

Random subset???

In [ ]:
random_color_images = []

for galname in results.keys():
    # print(galname)
    res = results[galname]
    node_pos = res["graph"].pos

    log_ages = res["log_ages"]
    gnn_preds = res["gnn_pred"]
    rf_preds = res["rf_pred"]

    origin = res["graph"].center
    radec = np.vstack([
        node_pos[:,0] / 3600 / np.cos(np.deg2rad(origin[1])) + origin[0],
        node_pos[:,1] / 3600 + origin[1]
    ]).T
    # print(radec[overestimates])
    # print((gnn_preds - log_ages)[overestimates])
    
    if galname in gal_image_name_mappings.keys():
        gal_image_name = gal_image_name_mappings[galname]     
        rng = np.random.RandomState(42)
        random_indices = rng.choice(radec.shape[0], size=4, replace=False)
        for ra_deg, dec_deg in tqdm.tqdm(radec[random_indices]):
            coord = SkyCoord(ra=ra_deg*u.degree, dec=dec_deg*u.degree, frame='icrs')
            ra_hms = coord.ra.to_string(unit=u.hour, sep=':')
            dec_dms = coord.dec.to_string(unit=u.deg, sep=':')
        
            
            input_files = [
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f814w_v1_exp-drc-sci.fits",
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f555w_v1_exp-drc-sci.fits",
                f"https://archive.stsci.edu/hlsps/phangs-hst/{gal_image_name}/hlsp_phangs-hst_hst_wfc3-uvis_{gal_image_name}_f438w_v1_exp-drc-sci.fits",
            ]
            
            center_coord = SkyCoord(f"{ra_deg} {dec_deg}", unit="deg")
            cutout_size = [64, 64]
            
            color_image = FITSCutout(input_files, center_coord, cutout_size).get_image_cutouts(colorize=True)[0]
            random_color_images.append(color_image)

In [ ]:
fig, axes = plt.subplots(4, 14, figsize=(14,4), dpi=300, )
for im, ax in zip(random_color_images, axes.flat):
    ax.imshow(im)
    ax.axis("off")
fig.subplots_adjust(left=0.02, wspace=0.02, hspace=0.02, top=0.98, bottom=0.02, right=0.98)


# Measuring residual error vs environment

In [ ]:
log_bins = np.array([0, 1, 2, 4, 8, 16, 32, 64, 128])
n_bootstrap = 300

r_link_kpc_thresholds = [0.02, 0.04, 0.08, 0.16]
fig, axes = plt.subplots(1, len(r_link_kpc_thresholds), figsize=(3*len(r_link_kpc_thresholds), 3), dpi=300, sharey=True)

for r_link_kpc_threshold, ax in zip(r_link_kpc_thresholds, axes.flat):
    residuals_gnn = []
    residuals_rf = []
    num_neighbors = []

    for galname in results.keys():
        res = results[galname]
    
        # convert edge separations to kpc, mask if greater than threshold, and finally number of edges from node (degree)
        galaxy_distance_kpc = res["graph"].u[:,1].item() * 1e3
        separation_arcsec = res["graph"].edge_attr[:, 0]
        separation_kpc = np.deg2rad(separation_arcsec / 3600) * galaxy_distance_kpc
        mask = separation_kpc < r_link_kpc_threshold
        filtered_edge_index = res["graph"].edge_index[:, mask]
        node_degree = degree(filtered_edge_index[0], num_nodes=res["graph"].num_nodes)
        
        node_pos = res["graph"].pos
        edge_list = res["graph"].edge_index
        
        log_ages = res["log_ages"]
        gnn_preds = res["gnn_pred"]
        rf_preds = res["rf_pred"]
    
        residuals_gnn.append(gnn_preds - log_ages)
        residuals_rf.append(rf_preds - log_ages)
        num_neighbors.append(node_degree)
    
    residuals_gnn = np.concatenate(residuals_gnn)
    residuals_rf = np.concatenate(residuals_rf)
    num_neighbors = np.concatenate(num_neighbors)
    
    
    # Precompute bin indices
    x_vals = num_neighbors
    bin_indices = np.digitize(x_vals, bins=log_bins) - 1
    n_bins = len(log_bins) - 1
    n_points = len(x_vals)
    
    # Generate bootstrap sample indices
    boot_indices = np.random.randint(0, n_points, size=(n_bootstrap, n_points))
    boot_bin_indices = bin_indices[boot_indices]

    def bootstrap_bin_means(y):
        boot_y = y[boot_indices]
        mask = ~np.isnan(boot_y)
        boot_y_filled = np.where(mask, boot_y, 0.0)
    
        sums = np.zeros((n_bootstrap, n_bins))
        counts = np.zeros((n_bootstrap, n_bins))
    
        for b in range(n_bins):
            in_bin = (boot_bin_indices == b)
            sums[:, b] = np.sum(boot_y_filled * in_bin, axis=1)
            counts[:, b] = np.sum(mask & in_bin, axis=1)
    
        means = sums / counts
        return (
            np.nanmean(means, axis=0),               
            np.nanpercentile(means, 16, axis=0),     
            np.nanpercentile(means, 84, axis=0)      
        )
    
    # Bootstrap for both RF and GNN
    mean_rf, low_rf, high_rf = bootstrap_bin_means(residuals_rf)
    mean_gnn, low_gnn, high_gnn = bootstrap_bin_means(residuals_gnn)
    
    # X positions = bin centers
    bin_centers = (log_bins[:-1] + log_bins[1:]) / 2 + 0.5
    
    # Plot with confidence bands
    ax.plot(bin_centers, mean_rf, c=c0, label="RF", lw=2, solid_capstyle='round')
    ax.fill_between(bin_centers, low_rf, high_rf, color=c0, alpha=0.2, lw=0)
    
    ax.plot(bin_centers, mean_gnn, c=c3, label="GNN", lw=2, solid_capstyle='round')
    ax.fill_between(bin_centers, low_gnn, high_gnn, color=c3, alpha=0.2, lw=0)

    if ax == axes[0]:
        ax.text(bin_centers[2]*1.1, mean_rf[2], "RF", color=c0, fontsize=16, va="center")
        ax.text(bin_centers[2]*1.1, mean_gnn[2], "GNN", color=c3, fontsize=16, va="center")
    
    ax.axhline(0, c='k', alpha=0.1)
    
    ax.set_xscale("log")
    ax.set_xticks([1, 3, 10, 30], [1, 3, 10, 30])
    ax.set_xlabel(r"$1 + N_{\rm neighbors}$", fontsize=12)
    ax.grid(alpha=0.15)
    ax.set_xlim(0.9, 35)
    ax.set_ylim(-0.45, 0.45)

    ax.text(0.04, 0.07, f"{r_link_kpc_threshold*1e3:g} pc", fontsize=16, transform=ax.transAxes)

axes[0].set_ylabel(r"Residual log(age/yr)", fontsize=12)

fig.subplots_adjust(wspace=0.04)

plt.savefig(results_dir / "figures/residuals-vs-n_neighbors.pdf", bbox_inches="tight")

In [ ]:

def to_numpy(x): return np.array(x)

def max_k_nn_separations(edge_index, separations, num_nodes, k=5, max_sep_kpc=None):
    ei0 = edge_index[0].astype(int)
    ei1 = edge_index[1].astype(int)
    mean_sep = np.full(num_nodes, np.nan, dtype=float)

    for node in range(num_nodes):
        mask = (ei0 == node) | (ei1 == node)
        if max_sep_kpc is not None:
            mask &= (separations <= max_sep_kpc)
        s = separations[mask]
        if s.size < 1:# or k if we want to omit them
            continue
        s_sorted = np.sort(s)
        mean_sep[node] = np.max(s_sorted[:min(k, s_sorted.size)])
    return mean_sep

def mean_k_nn_separations(edge_index, separations, num_nodes, k=10, max_sep_kpc=None):
    ei0 = edge_index[0].astype(int)
    ei1 = edge_index[1].astype(int)
    mean_sep = np.full(num_nodes, np.nan, dtype=float)

    for node in range(num_nodes):
        mask = (ei0 == node) | (ei1 == node)
        if max_sep_kpc is not None:
            mask &= (separations <= max_sep_kpc)
        s = separations[mask]
        if s.size < 1:# or k if we want to omit them
            continue
        s_sorted = np.sort(s)
        mean_sep[node] = np.mean(s_sorted[:min(k, s_sorted.size)])
    return mean_sep

def bootstrap_bin_means(y, bin_indices, n_bins, n_bootstrap=300):
    valid_mask = (bin_indices >= 0) & (~np.isnan(y))
    y = y[valid_mask]
    bin_indices = bin_indices[valid_mask].astype(int)
    n_points = len(y)
    if n_points == 0:
        return (np.full(n_bins, np.nan),
                np.full(n_bins, np.nan),
                np.full(n_bins, np.nan))

    boot_idx = np.random.randint(0, n_points, size=(n_bootstrap, n_points))
    boot_bin_idx = bin_indices[boot_idx]
    boot_y = y[boot_idx]
    mask = ~np.isnan(boot_y)

    sums = np.zeros((n_bootstrap, n_bins), dtype=float)
    counts = np.zeros((n_bootstrap, n_bins), dtype=float)

    for b in range(n_bins):
        in_bin = (boot_bin_idx == b)
        sums[:, b] = np.sum(boot_y * (in_bin & mask), axis=1)
        counts[:, b] = np.sum(in_bin & mask, axis=1)

    means = np.divide(sums, counts, out=np.full_like(sums, np.nan), where=(counts > 0))
    return (np.nanmean(means, axis=0),
            np.nanpercentile(means, 16, axis=0),
            np.nanpercentile(means, 84, axis=0))
    


In [ ]:
has_5_nodes = []
for galname in results.keys():
    node_degrees = degree(results[galname]["graph"].edge_index[0], results[galname]["graph"].num_nodes)
    has_5_nodes.append(node_degrees.float() > 5)

print("Fraction of galaxies that have > 5 nodes:", np.concatenate(has_5_nodes).mean())

In [ ]:
has_10_nodes = []
for galname in results.keys():
    node_degrees = degree(results[galname]["graph"].edge_index[0], results[galname]["graph"].num_nodes)
    has_10_nodes.append(node_degrees.float() > 10)

print("Fraction of galaxies that have > 10 nodes:", np.concatenate(has_10_nodes).mean())

In [ ]:
bins_kpc = np.logspace(np.log10(0.02), np.log10(2.0), num=9)
n_bootstrap = 1000

residuals_gnn_mean = []
residuals_rf_mean = []
seps_mean = []

for galname in results.keys():
    res = results[galname]
    galaxy_distance_kpc = res["graph"].u[:,1].item() * 1e3
    separation_arcsec = to_numpy(res["graph"].edge_attr[:, 0])
    separation_kpc = np.deg2rad(separation_arcsec / 3600.0) * galaxy_distance_kpc
    edge_index = to_numpy(res["graph"].edge_index)
    num_nodes = int(res["graph"].num_nodes)

    # mean of first 10 separations
    sep_mean = mean_k_nn_separations(edge_index, separation_kpc, num_nodes, k=10)

    log_ages = to_numpy(res["log_ages"])
    gnn_preds = to_numpy(res["gnn_pred"])
    rf_preds  = to_numpy(res["rf_pred"])


    residuals_gnn_mean.append(gnn_preds - log_ages)
    residuals_rf_mean.append(rf_preds - log_ages)
    seps_mean.append(sep_mean)

residuals_gnn_mean = np.concatenate(residuals_gnn_mean)
residuals_rf_mean  = np.concatenate(residuals_rf_mean)
seps_mean = np.concatenate(seps_mean)

bin_indices_mean = np.digitize(seps_mean, bins=bins_kpc) - 1
n_bins = len(bins_kpc) - 1

mean_rf, low_rf, high_rf = bootstrap_bin_means(residuals_rf_mean, bin_indices_mean, n_bins, n_bootstrap)
mean_gnn, low_gnn, high_gnn = bootstrap_bin_means(residuals_gnn_mean, bin_indices_mean, n_bins, n_bootstrap)

bin_centers = np.sqrt(bins_kpc[:-1] * bins_kpc[1:])

densities = 3 / (np.pi * bin_centers**2)

fig, ax = plt.subplots(1, 1, figsize=(3.75, 3.25), dpi=300)

ax.plot(densities, mean_rf, c=c0, ls="-", lw=2, solid_capstyle='round')
ax.fill_between(densities, low_rf, high_rf, color=c0, alpha=0.2, lw=0)
ax.plot(densities, mean_gnn, c=c3, ls="-", lw=2, solid_capstyle='round')
ax.fill_between(densities, low_gnn, high_gnn, color=c3, alpha=0.2, lw=0)

ax.axhline(0, c='k', alpha=0.12)
ax.set_xscale("log")
xticks = np.array([1, 10, 100, 1000])
# xticks = xticks[(xticks >= bins_kpc[0]) & (xticks <= bins_kpc[-1])]
ax.set_xticks(xticks)
ax.set_xticklabels([f"{t:g}" for t in xticks])
ax.set_xlim(0.3, 2000)
ax.set_ylim(-0.1,  0.4)
ax.set_xlabel(r"$\Sigma_{10}$ [kpc$^{-2}$]", fontsize=12)
ax.set_ylabel(r"Mean Residual [dex]", fontsize=12)

# show radius
ax2 = ax.twiny()
ax2.set_xscale("log")
xticks2 = [1000, 300, 100, 30]
ax2.set_xticks(xticks2)
ax2.set_xticklabels([f"{t:g}" for t in xticks2])
ax2.set_xlim(1000* np.sqrt(3 / (np.pi * 0.3)), 1000* np.sqrt(3 / (np.pi * 2000)))
ax2.set_xlabel("$d_{10}$ [pc]", fontsize=12)

ax.text(0.05, 0.88, "RF", color=c0, fontsize=18, va="center", ha="left", transform=ax.transAxes)
ax.text(0.05, 0.75, "GNN", color=c3, fontsize=18, va="center", ha="left", transform=ax.transAxes)
ax.grid(alpha=0.05)

for ax_ in [ax, ax2]:
    ax_.tick_params(axis="both", which="major", labelsize=10)
    ax_.minorticks_off()
plt.tight_layout()
plt.savefig(results_dir / "figures/residuals_vs_10nn-separations.pdf", bbox_inches="tight")
plt.show()

# Ablation study results

In [ ]:
experiment_names = [
    "gnn-best-sweep",
    "gnn-experiment-no_galaxy_distance",
    "gnn-experiment-no_inclination_angle",
    "gnn-experiment-no_graph_features",
    "gnn-experiment-no_graph-no_angles",
    "gnn-experiment-no_graph-no_separations",
    "gnn-experiment-no_galaxy-no_edge",
    # "gnn-experiment-only_magnitudes",
]

df_metrics = {}

for exp in experiment_names:
    df = pd.concat([pd.read_csv(results_dir / f"{exp}/cv_gnn_fold_{k}_predictions.csv") for k in range(5)])
    # print(exp)
    
    is_finite = np.isfinite(df["y_true"])
    p = df[f"y_pred"][is_finite]
    y = df[f"y_true"][is_finite]

    _metrics = {}
    for z, [metric, func] in enumerate(metrics_mapping.items()):
        # if metric == "Outlier Frac.": continue
        score = func(p, y)
        # print(f"{metric: >10s} = {score:.3f}")
        _metrics[metric] = score

    df_metrics[exp] = _metrics

df_metrics = pd.DataFrame.from_dict(df_metrics, orient="index")
df_metrics

# GNN age residuals vs Thilker+25 - Turner+21 age residuals

In [ ]:
import matplotlib
from scipy.stats import spearmanr, pearsonr


COMPARISON_DIR = Path(f"{base_dir}/results/results_T21-comparison")
figure_dir = Path(f"{base_dir}/results/figures")


# Load the merged GNN comparison data
gnn_merged = pd.read_csv(COMPARISON_DIR / "comparison_data.csv")

# Calculate deltas (if not already in the CSV)
gnn_merged["delta_pred_T21"] = gnn_merged["pred_T21"] - gnn_merged["target_T21"]
gnn_merged["delta_target_T25_T21"] = gnn_merged["target_T25"] - gnn_merged["target_T21"]

# drop NaNs for stats
gnn_merged_clean = gnn_merged.dropna(subset=["delta_target_T25_T21", "delta_pred_T21", "target_T25"])

rf_merged = pd.read_csv(COMPARISON_DIR / "comparison_data_rf.csv")
rf_merged_clean = rf_merged.dropna(subset=["delta_target_T25_T21", "delta_pred_T21", "target_T25"])



In [ ]:
# plot T21 age vs. predicted age

fig, axes = plt.subplots(1, 2, figsize=(7.5, 4), dpi=300, sharex=False, sharey=True)

colors = [c0, c3]
lims = (5.8, 10.3)
labels = ["RF (trained on T21)", "GNN (trained on T21)"]

for model, df, ax, color in zip(labels, [rf_merged_clean, gnn_merged_clean], axes.flat, colors):
    is_finite = np.isfinite(df["target_T21"])
    p = df[f"pred_T21"][is_finite]
    y = df[f"target_T21"][is_finite]

    cmap = LinearSegmentedColormap.from_list("my_cmap", ["#ffffff", color])
    
    # ax.scatter(y, p, s=0.3, alpha=1, color=color, linewidths=0, edgecolors="none", rasterized=True)
    ax.hist2d(y, p, bins=[42, 42], range=[(6, 10.2), (6, 10.2)], norm=LogNorm(vmin=1, vmax=3e2), cmap=cmap, rasterized=True)

    ax.plot(lims, lims, ls='-', c='w', lw=1.5)
    ax.plot(lims, lims, ls='-', c='0.5', lw=0.3)
    
    ax.set_xlim(*lims)
    ax.set_ylim(*lims)
    ax.set_xticks(range(*(map(lambda x: np.ceil(x).astype(int), lims))))
    ax.set_yticks(range(*(map(lambda x: np.ceil(x).astype(int), lims))))

    # ax.text(0.03, 0.9, model, fontsize=16, ha="left", transform=ax.transAxes)
    ax.set_title(model, fontsize=12, color=color, weight="bold")

    ax.set_aspect("equal")
    ax.grid(alpha=0.15)
    
    for z, [metric, func] in enumerate(metrics_mapping.items()):
        if metric in ["Outlier Frac.", "Bias", "$R^2$"]: continue
        score = func(p, y)
        ax.text(0.95, 0.06*(-0.2 + z), f"{metric: >7s} = {score:.3f}", fontsize=12, ha="right", transform=ax.transAxes)
    
        
fig.subplots_adjust(wspace=0.025, hspace=0.025, left=0.075, right=0.975, top=0.975, bottom=0.075)

axes[0].set_ylabel(r"Predicted log(age/yr)", fontsize=14)

axes[0].set_xlabel(r"Turner+21 log(age/yr)", fontsize=14)
axes[1].set_xlabel(r"Turner+21 log(age/yr)", fontsize=14)

fig.tight_layout()
plt.savefig(results_dir / "figures/T21_results_pred-vs-true.pdf")

# plt.clf()

In [ ]:
# plot T25 age vs. predicted age

fig, axes = plt.subplots(1, 2, figsize=(7.5, 4), dpi=300, sharex=False, sharey=True)

colors = [c0, c3]
lims = (5.8, 10.3)
labels = ["RF (trained on T21)", "GNN (trained on T21)"]

for model, df, ax, color in zip(labels, [rf_merged_clean, gnn_merged_clean], axes.flat, colors):
    is_finite = np.isfinite(df["target_T25"])
    p = df[f"pred_T21"][is_finite]
    y = df[f"target_T25"][is_finite]

    cmap = LinearSegmentedColormap.from_list("my_cmap", ["#ffffff", color])
    
    # ax.scatter(y, p, s=0.3, alpha=1, color=color, linewidths=0, edgecolors="none", rasterized=True)
    ax.hist2d(y, p, bins=[42, 42], range=[(6, 10.2), (6, 10.2)], norm=LogNorm(vmin=1, vmax=3e2), cmap=cmap, rasterized=True)

    ax.plot(lims, lims, ls='-', c='w', lw=1.5)
    ax.plot(lims, lims, ls='-', c='0.5', lw=0.3)
    
    ax.set_xlim(*lims)
    ax.set_ylim(*lims)
    ax.set_xticks(range(*(map(lambda x: np.ceil(x).astype(int), lims))))
    ax.set_yticks(range(*(map(lambda x: np.ceil(x).astype(int), lims))))

    # ax.text(0.03, 0.9, model, fontsize=16, ha="left", transform=ax.transAxes)
    ax.set_title(model, fontsize=12, color=color, weight="bold")

    ax.set_aspect("equal")
    ax.grid(alpha=0.15)
    
    for z, [metric, func] in enumerate(metrics_mapping.items()):
        if metric in ["Outlier Frac.", "Bias", "$R^2$"]: continue
        score = func(p, y)
        ax.text(0.95, 0.06*(-0.2 + z), f"{metric: >7s} = {score:.3f}", fontsize=12, ha="right", transform=ax.transAxes)
    
        
fig.subplots_adjust(wspace=0.025, hspace=0.025, left=0.075, right=0.975, top=0.975, bottom=0.075)

axes[0].set_ylabel(r"Predicted log(age/yr)", fontsize=14)

axes[0].set_xlabel(r"Thilker+25 log(age/yr)", fontsize=14)
axes[1].set_xlabel(r"Thilker+25 log(age/yr)", fontsize=14)

fig.tight_layout()
plt.savefig(results_dir / "figures/T21_results_pred-vs-T25.pdf")

# plt.clf()

In [ ]:

# # define splits
# bins = [
#     (r"age(T25) < 10 Myr", gnn_merged_clean["target_T25"] < 7),
#     (r"10 Myr $-$ 1 Gyr", (gnn_merged_clean["target_T25"] >= 7) & (gnn_merged_clean["target_T25"] <= 9)),
#     (r"> 1 Gyr", gnn_merged_clean["target_T25"] > 9)
# ]

# lims = [[-2, 2], [-2, 2], [-3, 4]]

# fig, axes = plt.subplots(1, 3, figsize=(12.5, 4), dpi=150)

# for ax, (label, mask), lim in zip(axes, bins, lims):
#     subset = gnn_merged_clean[mask]
    
#     x = subset["delta_target_T25_T21"]
#     y = subset["delta_pred_T21"]
    
#     spearman_corr, _ = spearmanr(x, y)
#     pearson_corr, _ = pearsonr(x, y)
#     count = len(x)

#     cmap = LinearSegmentedColormap.from_list("blue-ish", ["#fdfdfd", c3])
#     h = ax.hist2d(x, y, bins=41, cmap=cmap, norm=LogNorm(vmin=1, vmax=1e2), range=[lim, lim], cmin=1)
    
#     # limits
#     ax.set_xlim(lim)
#     ax.set_ylim(lim)
#     ax.axvline(x=0, color="k", alpha=0.1, lw=1)
#     ax.axhline(y=0, color="k", alpha=0.1, lw=1)
#     ax.grid(alpha=0.15)
#     ax.plot(lim, lim, "white", alpha=1, lw=1.5)
#     ax.plot(lim, lim, "k", alpha=1, lw=0.5)
    
#     ax.set_title(label, fontsize=16)
#     ax.set_xlabel(r"$\Delta {\rm age}$ (T25 SED $-$ T21 SED)", fontsize=16)
#     if ax == axes[0]:
#         ax.set_ylabel(r"$\Delta {\rm age}$ (T21 GNN $-$ T21 SED)", fontsize=16)
    
#     ax.text(0.05, 0.92, f"$\\rho =${spearman_corr:+.2f}", transform=ax.transAxes, fontsize=14, verticalalignment='center')
#     ax.text(0.05, 0.82, f"$r =${pearson_corr:+.2f}", transform=ax.transAxes, fontsize=14, verticalalignment='center')
#     ax.text(0.05, 0.72, f"$N = {count}$", transform=ax.transAxes, fontsize=14, verticalalignment='center')

#     ax.set_aspect("equal")

# plt.tight_layout()
# plt.subplots_adjust(right=0.9)
# cbar_ax = fig.add_axes([0.91, 0.115, 0.015, 0.825])
# fig.colorbar(h[3], cax=cbar_ax, label='Number of clusters')
# plt.savefig(figure_dir / "t21_gnn_vs_t25_3panel.pdf", bbox_inches='tight')

In [ ]:
# # Load the merged RF comparison data

# bins = [
#     (r"age(T25) < 10 Myr", rf_merged_clean["target_T25"] < 7),
#     (r"10 Myr $-$ 1 Gyr", (rf_merged_clean["target_T25"] >= 7) & (rf_merged_clean["target_T25"] <= 9)),
#     (r"> 1 Gyr", rf_merged_clean["target_T25"] > 9)
# ]

# lims = [[-2, 2], [-2, 2], [-3, 4]]

# fig, axes = plt.subplots(1, 3, figsize=(12.5, 4), dpi=150)

# for ax, (label, mask), lim in zip(axes, bins, lims):
#     subset = rf_merged_clean[mask]
    
#     x = subset["delta_target_T25_T21"]
#     y = subset["delta_pred_T21"]
    
#     spearman_corr, _ = spearmanr(x, y)
#     pearson_corr, _ = pearsonr(x, y)
#     count = len(x)

#     cmap = LinearSegmentedColormap.from_list("red-ish", ["#fdfdfd", c0])
#     h = ax.hist2d(x, y, bins=41, cmap=cmap, norm=LogNorm(vmin=1, vmax=1e2), range=[lim, lim], cmin=1)
    
#     ax.set_xlim(lim)
#     ax.set_ylim(lim)
#     ax.axvline(x=0, color="k", alpha=0.1, lw=1)
#     ax.axhline(y=0, color="k", alpha=0.1, lw=1)
#     ax.grid(alpha=0.15)
#     ax.plot(lim, lim, "white", alpha=1, lw=1.5)
#     ax.plot(lim, lim, "k", alpha=1, lw=0.5)
    
#     ax.set_title(label, fontsize=16)
#     ax.set_xlabel(r"$\Delta {\rm age}$ (T25 SED $-$ T21 SED)", fontsize=16)
#     if ax == axes[0]:
#         ax.set_ylabel(r"$\Delta {\rm age}$ (T21 RF $-$ T21 SED)", fontsize=16)
    
#     ax.text(0.05, 0.92, f"$\\rho =${spearman_corr:+.2f}", transform=ax.transAxes, fontsize=14, verticalalignment='center')
#     ax.text(0.05, 0.82, f"$r =${pearson_corr:+.2f}", transform=ax.transAxes, fontsize=14, verticalalignment='center')
#     ax.text(0.05, 0.72, f"$N = {count}$", transform=ax.transAxes, fontsize=14, verticalalignment='center')

#     ax.set_aspect("equal")

# plt.tight_layout()
# plt.subplots_adjust(right=0.9)
# cbar_ax = fig.add_axes([0.91, 0.115, 0.015, 0.825])
# fig.colorbar(h[3], cax=cbar_ax, label='Number of clusters')
# plt.savefig(figure_dir / "t21_rf_vs_t25_3panel.pdf", bbox_inches='tight')

## Color-color plots

In [ ]:
bc03 = {
    "1 Myr": (-0.3319526627218935, -1.647410358565737),
    "5 Myr": (-0.14260355029585797, -1.202191235059761),
    "10 Myr": (0.7686390532544378, -1.282868525896414),
    "30 Myr": (0.5485207100591716, -0.9810756972111554),
    "100 Myr": (0.45621301775147927, -0.5836653386454183),
    "500 Myr": (0.6597633136094674, 0.16932270916334657),
    "1 Gyr": (0.8822485207100592, 0.15139442231075695),
    "13.8 Gyr": (1.3650887573964496, 0.4770916334661355)
}

def overlay_bc03_tracks(ax, tracks=bc03, print_labels=False, reddening_vector=False, color='black', fontsize=10):
    """Plots BC03 age tracks as markers with text labels on the provided axis."""

    vi_colors = [coords[0] for coords in tracks.values()]
    ub_colors = [coords[1] for coords in tracks.values()]
    ax.plot(vi_colors, ub_colors, marker='o', color=color, markersize=3, zorder=10)
    
    for label, (vi, ub) in tracks.items():
        if print_labels:
            import matplotlib.patheffects as patheffects
            x_pos, y_pos = vi + 0.05, ub
            ha = 'left'
            
            if label == "500 Myr":
                xlim, ylim = ax.get_xlim(), ax.get_ylim()
                x_pos = vi - 0.02 * (xlim[1] - xlim[0])
                y_pos = ub - 0.02 * (ylim[1] - ylim[0])
                ha = 'right'

            ax.text(x_pos, y_pos, label, color=color, fontsize=fontsize, va='center', ha=ha, zorder=11,
                    path_effects=[patheffects.withStroke(linewidth=2, foreground='white')])

        if reddening_vector:
            # begin at V-I = 1.2, U-B = -1.6
            vi_start, ub_start = 1.2, -1.6
            d_vi, d_ub = 0.421, 0.275
            ax.annotate(
                "", 
                xy=(vi_start + d_vi, ub_start + d_ub), xytext=(vi_start, ub_start),
                arrowprops=dict(arrowstyle="->", color=color, lw=1)
            )
            xlim, ylim = ax.get_xlim(), ax.get_ylim()
            x_pos = vi_start - 0.01 * (xlim[1] - xlim[0])
            y_pos = ub_start + 0.01 * (ylim[1] - ylim[0])
            ax.text(x_pos, y_pos, "$A_V$ = 1 mag", fontsize=12)

In [ ]:
# plot U-B vs V-I color color diagrams, where markers are colored by 
# (1) Turner+21 SED age 
# (2) RF (trained on T21) predicted age, 
# (3) GNN (trained on T21) predicted age, and 
# (4) Thilker+25 SED age

# Load comparison data (both GNN and RF)
comparison_dir = base_dir / "results" / "results_T21-comparison"
gnn_data = pd.read_csv(comparison_dir / "comparison_data.csv")
rf_data = pd.read_csv(comparison_dir / "comparison_data_rf.csv")

# Merge GNN and RF predictions 
merged = pd.merge(
    gnn_data[["galaxy", "cluster_id", "pred_T21", "target_T21", "target_T25"]],
    rf_data[["galaxy", "cluster_id", "pred_T21"]],
    on=["galaxy", "cluster_id"],
    suffixes=("_gnn", "_rf")
)
merged = merged.rename(columns={"pred_T21_gnn": "age_gnn", "pred_T21_rf": "age_rf", 
                                 "target_T21": "age_T21", "target_T25": "age_T25"})

# Load photometry from the graphs
pyg_data_fname_t21 = base_dir / "data" / "processed" / "galaxy_graphs_T21.pkl"
with open(pyg_data_fname_t21, "rb") as f:
    data_dict_t21 = pickle.load(f)

# Extract photometry for each cluster
# features F275W, F336W, F438W, F555W, F814W, CI
phot_rows = []
for galaxy, graph in data_dict_t21.items():
    for i in range(graph.x.shape[0]):
        phot_rows.append({
            "galaxy": galaxy,
            "cluster_id": graph.cluster_id[i],
            "F275W": graph.x[i, 0].item(),
            "F336W": graph.x[i, 1].item(),
            "F438W": graph.x[i, 2].item(),
            "F555W": graph.x[i, 3].item(),
            "F814W": graph.x[i, 4].item(),
        })
phot_df = pd.DataFrame(phot_rows)

# Compute colors: U-B ~ F336W - F438W, V-I ~ F555W - F814W
phot_df["U_B"] = phot_df["F336W"] - phot_df["F438W"]
phot_df["V_I"] = phot_df["F555W"] - phot_df["F814W"]

# Merge with age data
plot_data = pd.merge(
    merged, 
    phot_df[["galaxy", "cluster_id", "U_B", "V_I"]], 
    on=["galaxy", "cluster_id"], how="inner"
)
plot_data = plot_data.dropna()

print(f"Using {len(plot_data)} clusters (T21 x T25)")

# 4-panel figure
fig, axes = plt.subplots(2, 2, figsize=(10, 9), dpi=300, sharex=True, sharey=True)
axes = axes.flatten()

age_cols = ["age_T21", "age_T25", "age_rf", "age_gnn"]
titles = ["Turner+21 SED Age", "Thilker+25 SED Age", "RF (trained on T21) Predicted Age", "GNN (trained on T21) Predicted Age"]

cmap = cmr.get_sub_cmap(cmr.torch, 0.2, 0.9, N=8)

for ax, age_col, title in zip(axes, age_cols, titles):
    sc = ax.scatter(
        plot_data["V_I"], 
        plot_data["U_B"], 
        c=plot_data[age_col], 
        s=10, 
        alpha=1,
        cmap=cmap,
        vmin=6,
        vmax=10,
        edgecolors="none",
        rasterized=True
    )
    if ax == axes[2] or ax == axes[3]:
        ax.set_xlabel(r"F555W $-$ F814W", fontsize=12)
    if ax == axes[0] or ax == axes[2]:
        ax.set_ylabel(r"F336W $-$ F438W", fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_xlim(-0.8, 2.2)
    ax.set_ylim(1.5, -2.1) # inverted
    ax.grid(alpha=0.15)

    overlay_bc03_tracks(ax, print_labels=ax==axes[0], reddening_vector=ax==axes[0])

# Add colorbar
plt.tight_layout()
plt.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.93, 0.068, 0.02, 0.89])
cbar = fig.colorbar(sc, cax=cbar_ax)
cbar.set_label(r"$\log({\rm age / yr})$", fontsize=12)

plt.savefig(figure_dir / "color_color_diagram_T21trained_4panel.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# color-color diagram but colored by *residuals* this time
# (i) T25 - T21
# (ii) RF(T21) - T21
# (iii) GNN(T21) - T21

# begin with same plot_data as before (T21 x T25), but add residuals
plot_data["age_resid_T25"] = plot_data["age_T25"] - plot_data["age_T21"]
plot_data["age_resid_rf"] = plot_data["age_rf"] - plot_data["age_T21"]
plot_data["age_resid_gnn"] = plot_data["age_gnn"] - plot_data["age_T21"]


# 3-panel figure (T25 - T21, RF - T21, GNN - T21)
fig, axes = plt.subplots(1, 3, figsize=(12, 4.5), dpi=300, sharex=True, sharey=True)

age_cols = ["age_resid_T25", "age_resid_rf", "age_resid_gnn"]
titles = ["T25 $-$ T21", "RF(T21) Residual", "GNN(T21) Residual"]

cmap = cmr.get_sub_cmap(cmr.fusion_r, 0.15, 0.85)

for ax, age_col, title in zip(axes, age_cols, titles):
    sc = ax.scatter(
        plot_data["V_I"], 
        plot_data["U_B"], 
        c=plot_data[age_col], 
        s=5*np.abs(plot_data[age_col]) + 0.1, 
        alpha=1,
        cmap=cmap,
        vmin=-1,
        vmax=1,
        edgecolors="none",
        rasterized=True
    )
    ax.set_xlabel(r"F555W $-$ F814W", fontsize=12)
    if ax == axes[0]:
        ax.set_ylabel(r"F336W $-$ F438W", fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_xlim(-0.8, 2.2)
    ax.set_ylim(1.5, -2.1) # inverted
    ax.grid(alpha=0.15)

    overlay_bc03_tracks(ax, reddening_vector=ax==axes[0])


# show key with abs(age residual) marker sizes
axes[0].legend(
    [matplotlib.lines.Line2D([0], [0], marker='o', linestyle='none', markerfacecolor='k', markeredgewidth=0, markersize=np.sqrt(5*s + 0.1)) for s in [0.5, 1, 3]],
    ["0.5", "1.0", "3.0"],
    title=r"$|\Delta \log({\rm age / yr})|$",
    framealpha=1,
    fontsize=12,
    title_fontsize=12,
)

plt.tight_layout()
plt.subplots_adjust(right=0.91)
cbar_ax = fig.add_axes([0.93, 0.135, 0.015, 0.78])
cbar = fig.colorbar(sc, cax=cbar_ax)
cbar.set_label(r"Residual $\log({\rm age / yr})$", fontsize=12)

plt.savefig(figure_dir / "color_color_diagram_T21residuals_3panel.pdf", bbox_inches="tight")
# plt.show()

In [ ]:
# color-color diagram but for ML models trained on T25


# Load T25-trained GNN and RF predictions (after running modified scripts)
gnn_t25_dir = base_dir / "results" / "gnn"
rf_t25_dir = base_dir / "results" / "rf"

# Concatenate fold predictions
gnn_preds_list = []
for i in range(5):
    df = pd.read_csv(gnn_t25_dir / f"cv_gnn_fold_{i}_predictions.csv")
    gnn_preds_list.append(df)
gnn_preds = pd.concat(gnn_preds_list, ignore_index=True)

rf_preds_list = []
for i in range(5):
    df = pd.read_csv(rf_t25_dir / f"cv_rf_fold_{i}_predictions.csv")
    rf_preds_list.append(df)
rf_preds = pd.concat(rf_preds_list, ignore_index=True)

# Merge GNN and RF predictions
merged = pd.merge(
    gnn_preds[["galaxy", "cluster_id", "y_pred", "y_true"]],
    rf_preds[["galaxy", "cluster_id", "y_pred"]],
    on=["galaxy", "cluster_id"],
    suffixes=("_gnn", "_rf")
)
merged = merged.rename(columns={
    "y_pred_gnn": "age_gnn", 
    "y_pred_rf": "age_rf", 
    "y_true": "age_T25"  # T25 is the ground truth for these models
})

# Load photometry from the T25 graphs
pyg_data_fname = base_dir / "data" / "processed" / "galaxy_graphs.pkl"
with open(pyg_data_fname, "rb") as f:
    data_dict = pickle.load(f)

# Extract photometry for each cluster
phot_rows = []
for galaxy, graph in data_dict.items():
    for i in range(graph.x.shape[0]):
        phot_rows.append({
            "galaxy": galaxy,
            "cluster_id": graph.cluster_id[i],
            "F275W": graph.x[i, 0].item(),
            "F336W": graph.x[i, 1].item(),
            "F438W": graph.x[i, 2].item(),
            "F555W": graph.x[i, 3].item(),
            "F814W": graph.x[i, 4].item(),
        })
phot_df = pd.DataFrame(phot_rows)

# Compute colors
phot_df["U_B"] = phot_df["F336W"] - phot_df["F438W"]
phot_df["V_I"] = phot_df["F555W"] - phot_df["F814W"]

# Merge with age data
plot_data = pd.merge(
    merged, 
    phot_df[["galaxy", "cluster_id", "U_B", "V_I"]], 
    on=["galaxy", "cluster_id"], 
    how="inner"
)
plot_data = plot_data.dropna()

print(f"Using {len(plot_data)} clusters (T25)")

# 3-panel figure (T25 SED, RF, GNN - no T21 here since models trained on T25)
fig, axes = plt.subplots(1, 3, figsize=(11.5, 4), dpi=300, sharex=True, sharey=True)

age_cols = ["age_T25", "age_rf", "age_gnn"]
titles = ["Thilker+25 SED Age", "RF (trained on T25) Pred.", "GNN (trained on T25) Pred."]

cmap = cmr.get_sub_cmap(cmr.torch, 0.2, 0.9, N=8)

for ax, age_col, title in zip(axes, age_cols, titles):
    sc = ax.scatter(
        plot_data["V_I"], 
        plot_data["U_B"], 
        c=plot_data[age_col], 
        s=7, 
        alpha=1,
        cmap=cmap,
        vmin=6,
        vmax=10,
        edgecolors="none",
        rasterized=True
    )
    ax.set_xlabel(r"F555W $-$ F814W", fontsize=12)
    if ax == axes[0]:
        ax.set_ylabel(r"F336W $-$ F438W", fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_xlim(-0.8, 2.2)
    ax.set_ylim(1.5, -2.1) # inverted
    ax.grid(alpha=0.15)
    overlay_bc03_tracks(ax, reddening_vector=ax==axes[0])


plt.tight_layout()
plt.subplots_adjust(right=0.91)
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.75])
cbar = fig.colorbar(sc, cax=cbar_ax)
cbar.set_label(r"$\log({\rm age / yr})$", fontsize=12)

plt.savefig(figure_dir / "color_color_diagram_T25trained_3panel.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# color-color diagram but colored by *residuals* for
# (i) RF(T25) - T25
# (ii) GNN(T25) - T25

# begin with same plot_data as before (T21 x T25), but add residuals
plot_data["age_resid_rf"] = plot_data["age_rf"] - plot_data["age_T25"]
plot_data["age_resid_gnn"] = plot_data["age_gnn"] - plot_data["age_T25"]

# 2-panel figure (RF - T21, GNN - T21)
fig, axes = plt.subplots(1, 2, figsize=(8.25, 4.5), dpi=300, sharex=True, sharey=True)

age_cols = ["age_resid_rf", "age_resid_gnn"]
titles = ["RF Residual", "GNN Residual"]

cmap = cmr.get_sub_cmap(cmr.fusion_r, 0.15, 0.85)

for ax, age_col, title in zip(axes, age_cols, titles):
    sc = ax.scatter(
        plot_data["V_I"], 
        plot_data["U_B"], 
        c=plot_data[age_col], 
        s=5*np.abs(plot_data[age_col])+0.1, 
        alpha=1,
        cmap=cmap,
        vmin=-1,
        vmax=1,
        edgecolors="none",
        rasterized=True,
    )
    if ax == axes[0]:
        ax.set_ylabel(r"F336W $-$ F438W", fontsize=12)
    ax.set_xlabel(r"F555W $-$ F814W", fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_xlim(-0.8, 2.2)
    ax.set_ylim(1.5, -2.1) # inverted
    ax.grid(alpha=0.15)
    overlay_bc03_tracks(ax)
    # overlay_bc03_tracks(ax, print_labels=ax==axes[0], reddening_vector=ax==axes[0])


# show key with abs(age residual) marker sizes
axes[0].legend(
    [matplotlib.lines.Line2D([0], [0], marker='o', linestyle='none', markerfacecolor='k', markeredgewidth=0, markersize=np.sqrt(5*s+0.1)) for s in [0.5, 1, 3]],
    ["0.5", "1.0", "3.0"],
    title=r"$|\Delta \log({\rm age / yr})|$",
    framealpha=1,
    fontsize=12,
    title_fontsize=12,
)

plt.tight_layout()
plt.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.92, 0.13, 0.02, 0.79])
cbar = fig.colorbar(sc, cax=cbar_ax)
cbar.set_label(r"Residual $\log({\rm age / yr})$", fontsize=12)

plt.savefig(figure_dir / "color_color_diagram_T25residuals_2panel.pdf", bbox_inches="tight")
# plt.show()

In [ ]:
# just thilker+25 age
fig, ax = plt.subplots(1, 1, figsize=(4.5, 4.5), dpi=300)

age_cols = ["age_T25"]
titles = ["Thilker+25 SED Age"]

cmap = cmr.get_sub_cmap(cmr.torch, 0.2, 0.9, N=8)

for age_col, title in zip(age_cols, titles):
    sc = ax.scatter(
        plot_data["V_I"], 
        plot_data["U_B"], 
        c=plot_data[age_col], 
        s=7, 
        alpha=1,
        cmap=cmap,
        vmin=6,
        vmax=10,
        edgecolors="none",
        rasterized=True
    )
    ax.set_xlabel(r"F555W $-$ F814W", fontsize=12)
    ax.set_ylabel(r"F336W $-$ F438W", fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.set_xlim(-0.8, 2.2)
    ax.set_ylim(1.5, -2.1) # inverted
    ax.grid(alpha=0.15)
    overlay_bc03_tracks(ax, reddening_vector=True, print_labels=True)


plt.tight_layout()
plt.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.91, 0.135, 0.03, 0.78])
cbar = fig.colorbar(sc, cax=cbar_ax)
cbar.set_label(r"$\log({\rm age / yr})$", fontsize=12)

plt.savefig(figure_dir / "color_color_diagram_T25_1panel.pdf", bbox_inches="tight")
plt.show()